# 05 · Feature Engineering

**Stage 5 of 8** · ACCESS → WRANGLE → EXPLORE → VISUALIZE → ENGINEER → MODEL → EVALUATE → RECOMMEND

> **Project premise:** if someone has had cancer and was treated, **will it come back?** And for every factor linked to it coming back: **correlation or causation?**
>
> All data here is public: a k-anonymous feature table (every combination of site group, region, sex and diagnosis period is shared by ≥ 3 patients), synthetic patients, and aggregate reports. The raw database and clinical notes stay private.

**Goal of this stage:** turn free-text clinical notes into model-ready variables, define the target correctly, and check for leakage.

**Business question:** Which variables describe a patient *before* their cancer could come back, and how do we define 'came back' fairly?

In [1]:
%matplotlib inline
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
PUBLIC = ROOT / "data" / "public" / "patients_features_public.csv"
SYNTH = ROOT / "data" / "synthetic" / "synthetic_patients.csv"
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "figure.dpi": 110})
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
df = pd.read_csv(PUBLIC)

### Q1 · Where do the features come from?
Most of the useful facts were **not** in database columns. They were written in free-text notes. `src/extract_features.py`
uses plain regular expressions (auditable by a clinician) to create 60 features. Demonstrated on **made-up** notes:

In [2]:
from extract_features import classify_site, receptor, grade, recurrence_event, metastatic_at_diagnosis, has_not_negated
notes = [
    "Ca breast diagnosed 2015, IDC grade 3, ER+ PR+ HER2 negative, mastectomy, FEC 6, RT, tamoxifen. No evidence of mets.",
    "Ca rectum 2016, operated, 12 cycles FOLFOX, developed liver mets in 2019, on xeloda.",
    "Ca ovary with liver mets, newly diagnosed 2021, plan for chemotherapy.",
]
pd.DataFrame([{
    "site": classify_site("", n)[1], "grade": grade(n), "ER": receptor(n, "er"), "HER2": receptor(n, "her2"),
    "liver mets (negation-aware)": has_not_negated(r"liver|mets", n),
    "metastatic at diagnosis": metastatic_at_diagnosis("", n), "came back": recurrence_event(n),
} for n in notes], index=["note 1", "note 2", "note 3"])

,site,grade,ER,HER2,liver mets (negation-aware),metastatic at diagnosis,came back
note 1,Breast,3.0,1.0,0.0,0,0,0
note 2,Colorectal,NaN,NaN,NaN,1,0,1
note 3,Ovary,NaN,NaN,NaN,1,1,0


Note the three different situations: note 1 is **disease-free** ("No evidence of mets" must not count as metastasis),
note 2 **came back** after treatment, and note 3 was **already metastatic at diagnosis** (a different outcome).

### Q2 · Engineered features used in this project

In [3]:
groups = {
    "Cancer & biology": ["site_group", "histology", "grade", "er", "pr", "her2", "breast_subtype"],
    "Spread": ["metastatic_at_dx", "mets_bone", "mets_liver", "mets_lung", "mets_brain", "n_met_sites"],
    "Treatment": ["surgery", "chemotherapy", "neoadjuvant_chemo", "radiotherapy", "hormonal_therapy", "anti_her2"],
    "Access & course": ["region", "treated_abroad", "missed_followup_or_delay", "years_since_dx", "n_visits"],
    "Outcome": ["eligible_for_relapse", "relapse", "rfs_years"],
}
pd.DataFrame([(g, c, df[c].notna().mean()) for g, cs in groups.items() for c in cs],
             columns=["group", "feature", "non-missing"]).style.format({"non-missing": "{:.0%}"})

,group,feature,non-missing
0,Cancer & biology,site_group,100%
1,Cancer & biology,histology,100%
2,Cancer & biology,grade,28%
3,Cancer & biology,er,17%
4,Cancer & biology,pr,17%
5,Cancer & biology,her2,16%
6,Cancer & biology,breast_subtype,54%
7,Spread,metastatic_at_dx,100%
8,Spread,mets_bone,100%
9,Spread,mets_liver,100%


### Q3 · The target: defining "came back" fairly (the most important decision in the project)
`relapse` = 1 if the note documents recurrence or new spread **after** treatment. It is defined only for the
**at-risk group**: malignant, not metastatic at diagnosis, treated, diagnosed ≥ 1 year before the note.

⚠️ **Lesson from version 1.** The first version required *curative* treatment only from patients **without**
relapse. Every untreated patient left in the group was therefore a relapse, which made every treatment look
protective (hormonal therapy OR 0.18) and the model look good (AUC 0.71). With **one rule for everyone**, the group is
138 patients and the effects shrink. Selection bias doesn't raise an error; it produces a nice chart.

In [4]:
at_risk = df[df.eligible_for_relapse == 1]
print(f"at-risk: {len(at_risk)}, relapses: {int(at_risk.relapse.sum())} ({at_risk.relapse.mean():.0%})")

at-risk: 138, relapses: 30 (22%)


### Q4 · Encoding categorical variables

In [5]:
from model import load
X, y = load(PUBLIC)
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(handle_unknown="ignore", min_frequency=3).fit(X[["site_group"]])
print(list(enc.get_feature_names_out()))

['site_group_Breast', 'site_group_CNS', 'site_group_Colorectal / anal', 'site_group_Genitourinary', 'site_group_Gynaecologic', 'site_group_Head & neck', 'site_group_Sarcoma', 'site_group_Skin', 'site_group_Upper GI / hepatobiliary', 'site_group_infrequent_sklearn']


Rare categories (< 3 patients) are pooled into "infrequent", so a model can't memorise one patient.

### Q5 · Grouping a numeric variable into meaningful bands

In [6]:
bands = pd.cut(at_risk.rfs_years, [0, 2, 5, 10.1], labels=["< 2 yrs", "2–5 yrs", "5+ yrs"], include_lowest=True)
at_risk.groupby(bands, observed=True).relapse.agg(["size", "mean"]).style.format({"mean": "{:.0%}"})

,size,mean
rfs_years,,
< 2 yrs,57,26%
2–5 yrs,32,22%
5+ yrs,49,16%


### Q6–Q7 · Which variables are allowed as predictors? The leakage check
A feature is allowed only if it is known **before** the cancer could come back. These are excluded because they
are **consequences** of relapse, and including them would let the model "predict" the past:

In [7]:
from model import LEAKY, CATEGORICAL, NUMERIC, BINARY
pd.DataFrame({"allowed predictors": pd.Series(CATEGORICAL + NUMERIC + BINARY),
              "excluded as leakage": pd.Series(LEAKY)}).fillna("")

,allowed predictors,excluded as leakage
0,site_group,recurrence_or_progression
1,breast_subtype,recurrence_year
2,region,metastatic
3,years_since_dx,n_met_sites
4,grade_known,mets_bone
5,grade3,mets_liver
6,surgery,mets_lung
7,radiotherapy,mets_brain
8,hormonal_therapy,mets_lymph_node
9,neoadjuvant_chemo,lines_of_chemo_2plus


## So what?

Feature engineering decided the answer more than any model did. The target definition (who is "at risk")
changed the headline result, and removing leaky features is what keeps the model honest.